<a href="https://colab.research.google.com/github/Arvind-NITCG/LLM/blob/main/LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [41]:
!nvidia-smi
!pip install PyPDF2
!pip install tiktoken

Sun May 17 14:10:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   76C    P0             34W /   70W |    3097MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [42]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import urllib.request
import tiktoken
import unicodedata
import re
import PyPDF2

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: cuda


In [43]:
pdf_path = 'Bhagavad-gita_As_It_Is english.pdf'
text = ""

print("1. Extracting raw text from PDF...")
with open(pdf_path, 'rb') as file:
    reader = PyPDF2.PdfReader(file)
    for page_num in range(len(reader.pages)):
        text += reader.pages[page_num].extract_text() + "\n"

print("2. Assassinating Copyright Footers...")
text = re.sub(r'Copyright.*?All Rights Reserved\.', '', text, flags=re.IGNORECASE | re.DOTALL)
text = re.sub(r'Copyright 199.*?Trust Int\'l\.', '', text, flags=re.IGNORECASE)

print("3. Executing Path A: Slicing pure English Purports...")
purport_sections = re.findall(r'PURPORT(.*?)(?=TEXT \d+|CHAPTER \d+|$)', text, re.DOTALL | re.IGNORECASE)
english_text = "\n".join(purport_sections)

print("4. Sanitizing the final dataset...")
clean_text = ''.join(c for c in unicodedata.normalize('NFD', english_text) if unicodedata.category(c) != 'Mn')
clean_text = re.sub(r'[^a-zA-Z0-9 \n.,!?;:\'\"()[\]{}-]', '', clean_text)
clean_text = re.sub(r'\n+', '\n', clean_text)
clean_text = re.sub(r' +', ' ', clean_text)

print(f"\nSUCCESS: Extracted {len(clean_text)} characters of pure, uninterrupted English philosophy!")

print("\n5. Executing Path B: Upgrading to BPE Tokenizer...")

enc = tiktoken.get_encoding("gpt2")
tokens = enc.encode(clean_text)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
data = torch.tensor(tokens, dtype=torch.long).to(device)

vocab_size = enc.n_vocab

print(f"New BPE Vocab Size: {vocab_size}")
print(f"Total Tokens for Training: {len(data)}")

# Split data into train and validation sets
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

1. Extracting raw text from PDF...
2. Assassinating Copyright Footers...
3. Executing Path A: Slicing pure English Purports...
4. Sanitizing the final dataset...

SUCCESS: Extracted 1653034 characters of pure, uninterrupted English philosophy!

5. Executing Path B: Upgrading to BPE Tokenizer...
New BPE Vocab Size: 50257
Total Tokens for Training: 473931


In [44]:
class PositionalEncoding(nn.Module):
    def __init__(self,d_model: int , max_len: int = 5000):
        super().__init__()
        pe = torch.zeros(max_len , d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return x

In [45]:
class FeedForward(nn.Module):

    def __init__(self, d_model: int, d_ff: int = 2048):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.linear2(self.relu(self.linear1(x)))

In [46]:
class Multihead_mask_attention(nn.Module):
    def __init__(self,model_dimension,numberofhead):
        super().__init__()
        assert model_dimension % numberofhead == 0

        self.model_dimension = model_dimension
        self.numberofhead = numberofhead
        self.d_k = model_dimension//numberofhead
        self.W_q = nn.Linear(model_dimension,model_dimension)
        self.W_k = nn.Linear(model_dimension,model_dimension)
        self.W_v = nn.Linear(model_dimension,model_dimension)

        self.W_o = nn.Linear(model_dimension,model_dimension)

    def forward(self,x,mask=None):
        batch_size , seq_len , _ = x.size()
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        Q = Q.view(batch_size,seq_len,self.numberofhead,self.d_k).transpose(1,2)
        K = K.view(batch_size,seq_len,self.numberofhead,self.d_k).transpose(1,2)
        V = V.view(batch_size,seq_len,self.numberofhead,self.d_k).transpose(1,2)

        score = torch.matmul(Q,K.transpose(-2,-1))
        score = score/math.sqrt(self.d_k)

        if mask is not None:
            score = score.masked_fill(mask==0 , -1e9)

        attn_probs = F.softmax(score , dim=-1)

        out = torch.matmul(attn_probs,V)
        out = out.transpose(1,2).contiguous()
        out = out.view(batch_size,seq_len,self.model_dimension)

        out = self.W_o(out)

        return out

In [47]:
class TransformerBlock(nn.Module):
    def __init__(self, model_dimension: int, numberofhead: int, dropout_rate: float = 0.1):
        super().__init__()

        self.attention = Multihead_mask_attention(model_dimension, numberofhead)
        self.feed_forward = FeedForward(model_dimension)

        self.norm1 = nn.LayerNorm(model_dimension)
        self.norm2 = nn.LayerNorm(model_dimension)

        self.dropout1 = nn.Dropout(dropout_rate)
        self.dropout2 = nn.Dropout(dropout_rate)

    def forward(self, x, mask=None):

        normalized_x = self.norm1(x)
        attention_out = self.attention(normalized_x, mask)
        x = x + self.dropout1(attention_out)

        normalized_x_2 = self.norm2(x)
        ff_out = self.feed_forward(normalized_x_2)
        x = x + self.dropout2(ff_out)

        return x

In [51]:
class GitaLLM(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, n_heads: int, n_layers: int, max_seq_len: int = 5000, dropout_rate: float = 0.2):
        super().__init__()

        self.token_embedding = nn.Embedding(vocab_size, d_model)

        self.positional_encoding = PositionalEncoding(d_model, max_seq_len)


        self.dropout = nn.Dropout(dropout_rate)

        self.blocks = nn.ModuleList([

            TransformerBlock(d_model, n_heads, dropout_rate) for _ in range(n_layers)
        ])

        self.final_norm = nn.LayerNorm(d_model)

        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, idx, targets=None):

        B, T = idx.size()
        mask = torch.tril(torch.ones(T, T)).to(idx.device)

        x = self.token_embedding(idx)
        x = self.positional_encoding(x)
        x = self.dropout(x)

        for block in self.blocks:
            x = block(x, mask=mask)

        x = self.final_norm(x)
        logits = self.lm_head(x)

        return logits

In [52]:
batch_size = 32
seq_len = 64
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - seq_len, (batch_size,))
    x = torch.stack([d[i:i+seq_len] for i in ix])
    y = torch.stack([d[i+1:i+seq_len+1] for i in ix])

    return x.to(device), y.to(device)
xb, yb = get_batch('train')
print(f"Input batch shape: {xb.shape}")
print(f"Target batch shape: {yb.shape}")

Input batch shape: torch.Size([32, 64])
Target batch shape: torch.Size([32, 64])


In [53]:
model = GitaLLM(vocab_size=vocab_size, d_model=384, n_heads=6, n_layers=4, dropout_rate=0.2).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-1)
max_iters = 1000
eval_interval = 300

print("Starting training...")

for iter in range(max_iters):
    if iter % eval_interval == 0:
        model.eval()
        with torch.no_grad():
            losses = []
            for _ in range(50):
                X, Y = get_batch('val')
                logits = model(X)
                loss = F.cross_entropy(logits.view(-1, vocab_size), Y.view(-1))
                losses.append(loss.item())
            val_loss = sum(losses) / len(losses)
            print(f"Step {iter}: Validation Loss = {val_loss:.4f}")
        model.train()
    xb, yb = get_batch('train')

    logits = model(xb)
    loss = F.cross_entropy(logits.view(-1, vocab_size), yb.view(-1))

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(f"Training complete! Final Loss: {loss.item():.4f}")

Starting training...
Step 0: Validation Loss = 10.9742
Step 300: Validation Loss = 6.8191
Step 600: Validation Loss = 6.6923
Step 900: Validation Loss = 6.8882
Training complete! Final Loss: 3.6163


In [55]:
def generate(model, start_str="Arjuna asked:How will I kill my grandfather and uncles krishna?\n", max_new_tokens=100000, temperature=0.8):
    model.eval()
    context = torch.tensor(enc.encode(start_str), dtype=torch.long, device=device).unsqueeze(0)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            context_cond = context[:, -seq_len:]
            logits = model(context_cond)
            logits = logits[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            context = torch.cat((context, idx_next), dim=1)
    generated_text = enc.decode(context[0].tolist())
    print(f"\n--- Generated Text ---\n{generated_text}")
    model.train()
generate(model,start_str= "Arjuna asked:How will I kill my grandfather and uncles Krishna?\n")


--- Generated Text ---
Arjuna asked:How will I kill my grandfather and uncles Krishna?
Generally.
In this not know that he is to act in that the Supreme Brahman
Lord He is neither, and cannot attain the spirit of all the universe. In the
Supreme Personality of Godhead as the Lord.
but we are the clutches of the Supreme Lord. He is the Lord, and He is
of the body of the Supreme Lords Godhead. This is concerned, within his mind,
ignor of his repentanceual material nature. Therefore he is the Supreme Lord.
no sphere of the Supersoul, he may be the Supreme Lord. These
position of Kanea as the Vedas and matter of the Lord is also Kanea. He is the
devotee of the body, for the transcendentities of all causes.
TEXT 14
YaMaSaJYaQa c Maa PaXYaNaaYa iNaaTa" )
Yaae YaaeSa-aeGaTMaaMYaR c YaSa ivk-Ma( )) 2 ))
knya-saitau nican
SYNONYMS
atmanauam of birth; dhama; tatthat; purunam of the great;
abhavau intelligence; aham I; caalso; aham I am;
mama the mode of passion; acarnam apraya by ignorance; nan